# Toronto Fire Incidents Data Cleaning Notebook

This notebook performs comprehensive data cleaning on the Toronto Fire Incidents CSV dataset.
It handles datetime parsing, categorical cleaning, missing values, outliers, and geolocation validation.

**Dataset:** 36,564 rows × 43 columns

## Setup: Import Required Libraries
Import pandas and numpy for data manipulation and analysis.

In [19]:
import pandas as pd
from datetime import datetime

## 1. Load & Explore

Load the raw CSV file and perform initial exploratory analysis to understand the structure, data types, and identify missing values before cleaning.

In [20]:
print("\n" + "="*80)
print("1. LOADING & EXPLORING DATA")
print("="*80)

# Load the CSV file
df = pd.read_csv('../data/fire_incidents_data.csv')

# Print basic information
print(f"\nDataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nData Types:\n{df.dtypes}")


1. LOADING & EXPLORING DATA

Dataset Shape: 36564 rows × 43 columns

Data Types:
_id                                                                int64
Incident_Number                                                      str
Initial_CAD_Event_Type                                               str
Final_Incident_Type                                                  str
Exposures                                                        float64
Incident_Station_Area                                             object
Incident_Ward                                                    float64
Intersection                                                         str
Latitude                                                         float64
Longitude                                                        float64
Property_Use                                                         str
Building_Status                                                      str
TFS_Alarm_Time                            

C:\Users\Ayush\AppData\Local\Temp\ipykernel_6240\3814309509.py:6: DtypeWarning: Columns (0: Incident_Station_Area) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/fire_incidents_data.csv')


In [21]:
# Print null counts for every column
print("\nNull Counts by Column:")
null_counts = df.isnull().sum()
null_percentages = (null_counts / len(df)) * 100
null_info = pd.DataFrame({
    'Null_Count': null_counts,
    'Percentage': null_percentages
})
print(null_info)


Null Counts by Column:
                                                    Null_Count  Percentage
_id                                                          0    0.000000
Incident_Number                                              0    0.000000
Initial_CAD_Event_Type                                       1    0.002735
Final_Incident_Type                                          0    0.000000
Exposures                                                25418   69.516464
Incident_Station_Area                                        1    0.002735
Incident_Ward                                              158    0.432119
Intersection                                                 2    0.005470
Latitude                                                     2    0.005470
Longitude                                                    2    0.005470
Property_Use                                               224    0.612624
Building_Status                                          18483   50.549721
T

In [22]:
# Print 5 sample rows
print("\nFirst 5 Sample Rows:")
print(df.head())


First 5 Sample Rows:
   _id Incident_Number        Initial_CAD_Event_Type  \
0    1       F18020956                  Vehicle Fire   
1    2       F18020969          Fire - Grass/Rubbish   
2    3       F18021182  Fire -  Highrise Residential   
3    4       F18021192  Fire - Commercial/Industrial   
4    5       F18021271            Fire - Residential   

                                 Final_Incident_Type  Exposures  \
0                                          01 - Fire        NaN   
1                                          01 - Fire        NaN   
2  03 - NO LOSS OUTDOOR fire (exc: Sus.arson,vand...        NaN   
3                                          01 - Fire        NaN   
4  03 - NO LOSS OUTDOOR fire (exc: Sus.arson,vand...        NaN   

  Incident_Station_Area  Incident_Ward                    Intersection  \
0                   441            1.0     Dixon Rd / 427 N Dixon Ramp   
1                   116           18.0  Sheppard Ave E / Clairtrell Rd   
2               

## 2. Parse Datetime Columns

Parse the datetime columns and compute derived time-based features:
- `response_time_min`: time from alarm to arrival
- `control_time_min`: time from alarm to fire control
- Temporal features: year, month, hour_of_day, day_of_week

In [23]:
print("\n" + "="*80)
print("2. PARSING DATETIME COLUMNS & COMPUTING DERIVED FEATURES")
print("="*80)

# Define datetime columns to parse
datetime_columns = [
    'TFS_Alarm_Time',
    'TFS_Arrival_Time',
    'Ext_agent_app_or_defer_time',
    'Fire_Under_Control_Time',
    'Last_TFS_Unit_Clear_Time'
]

# Parse datetime columns
for col in datetime_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')
    print(f"Parsed {col}")


2. PARSING DATETIME COLUMNS & COMPUTING DERIVED FEATURES
Parsed TFS_Alarm_Time
Parsed TFS_Arrival_Time
Parsed Ext_agent_app_or_defer_time
Parsed Fire_Under_Control_Time
Parsed Last_TFS_Unit_Clear_Time


In [24]:
# Compute response_time_min (arrival time - alarm time in minutes)
df['response_time_min'] = (df['TFS_Arrival_Time'] - df['TFS_Alarm_Time']).dt.total_seconds() / 60
print(f"Computed response_time_min (min={df['response_time_min'].min():.2f}, max={df['response_time_min'].max():.2f})")

# Compute control_time_min (fire under control time - alarm time in minutes)
df['control_time_min'] = (df['Fire_Under_Control_Time'] - df['TFS_Alarm_Time']).dt.total_seconds() / 60
print(f"Computed control_time_min (min={df['control_time_min'].min():.2f}, max={df['control_time_min'].max():.2f})")

Computed response_time_min (min=-37.42, max=682.37)
Computed control_time_min (min=0.32, max=3116.70)


In [25]:
# Extract temporal features from TFS_Alarm_Time
df['year'] = df['TFS_Alarm_Time'].dt.year
df['month'] = df['TFS_Alarm_Time'].dt.month
df['hour_of_day'] = df['TFS_Alarm_Time'].dt.hour
df['day_of_week'] = df['TFS_Alarm_Time'].dt.dayofweek  # Monday=0, Sunday=6

print(f"\nExtracted temporal features: year, month, hour_of_day, day_of_week")
print(f"Year range: {df['year'].min()} - {df['year'].max()}")
print(f"Months: {sorted(df['month'].unique())}")
print(f"Hours: {sorted(df['hour_of_day'].unique())}")


Extracted temporal features: year, month, hour_of_day, day_of_week
Year range: 2011 - 2024
Months: [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12)]
Hours: [np.int32(0), np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12), np.int32(13), np.int32(14), np.int32(15), np.int32(16), np.int32(17), np.int32(18), np.int32(19), np.int32(20), np.int32(21), np.int32(22), np.int32(23)]


## 3. Clean Categorical Columns

Strip numeric code prefixes (e.g., "01 - Fire" → "Fire") from categorical columns that contain these prefixes. This standardizes the categorical values.

In [38]:
print("\n" + "="*80)
print("3. CLEANING CATEGORICAL COLUMNS (REMOVING NUMERIC PREFIXES)")
print("="*80)

# Define categorical columns with numeric code prefixes
categorical_columns = [
    'Final_Incident_Type',
    'Initial_CAD_Event_Type',
    'Property_Use',
    'Area_of_Origin',
    'Possible_Cause',
    'Method_Of_Fire_Control',
    'Extent_Of_Fire',
    'Ignition_Source',
    'Material_First_Ignited',
    'Smoke_Alarm_at_Fire_Origin',
    'Sprinkler_System_Presence',
    'Status_of_Fire_On_Arrival'
]

# Process each categorical column
for col in categorical_columns:
    if col in df.columns:
        # Split by ' - ' and take everything after the prefix
        df[col] = df[col].str.split(' - ').str[1:].str.join(' - ')
        print(f"Cleaned {col}")
    else:
        print(f"Warning: Column '{col}' not found in dataset")


3. CLEANING CATEGORICAL COLUMNS (REMOVING NUMERIC PREFIXES)
Cleaned Final_Incident_Type
Cleaned Initial_CAD_Event_Type
Cleaned Property_Use
Cleaned Area_of_Origin
Cleaned Possible_Cause
Cleaned Method_Of_Fire_Control
Cleaned Extent_Of_Fire
Cleaned Ignition_Source
Cleaned Material_First_Ignited
Cleaned Smoke_Alarm_at_Fire_Origin
Cleaned Sprinkler_System_Presence
Cleaned Status_of_Fire_On_Arrival


## 4. Handle Missing Values

Address missing values appropriately based on the column type:
- Numeric casualty/count columns: fill with 0
- Numeric loss column: fill with median
- Categorical columns: fill with "Unknown"
- Flag columns with > 40% nulls as potential data quality issues.

In [27]:
print("\n" + "="*80)
print("4. HANDLING MISSING VALUES")
print("="*80)

# Define columns for numeric null filling
numeric_zero_fill_columns = [
    'Civilian_Casualties',
    'TFS_Firefighter_Casualties',
    'Count_of_Persons_Rescued',
    'Estimated_Number_Of_Persons_Displaced'
]

# Fill numeric casualty/count columns with 0
for col in numeric_zero_fill_columns:
    if col in df.columns and df[col].isnull().sum() > 0:
        df[col].fillna(0, inplace=True)
        print(f"Filled {col} nulls with 0 ({df[col].isnull().sum()} nulls remaining)")


4. HANDLING MISSING VALUES
Filled Civilian_Casualties nulls with 0 (10697 nulls remaining)
Filled TFS_Firefighter_Casualties nulls with 0 (222 nulls remaining)
Filled Count_of_Persons_Rescued nulls with 0 (223 nulls remaining)
Filled Estimated_Number_Of_Persons_Displaced nulls with 0 (18485 nulls remaining)


C:\Users\Ayush\AppData\Local\Temp\ipykernel_6240\3539990565.py:16: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df[col].fillna(0, inplace=True)


In [28]:
# Fill Estimated_Dollar_Loss with median
if 'Estimated_Dollar_Loss' in df.columns and df['Estimated_Dollar_Loss'].isnull().sum() > 0:
    median_loss = df['Estimated_Dollar_Loss'].median()
    df['Estimated_Dollar_Loss'].fillna(median_loss, inplace=True)
    print(f"Filled Estimated_Dollar_Loss nulls with median value {median_loss:.2f}")

Filled Estimated_Dollar_Loss nulls with median value 3000.00


C:\Users\Ayush\AppData\Local\Temp\ipykernel_6240\3836613617.py:4: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['Estimated_Dollar_Loss'].fillna(median_loss, inplace=True)


In [29]:
# Fill all remaining categorical columns with "Unknown"
categorical_nulls_filled = 0
for col in df.columns:
    if df[col].dtype == 'object' and df[col].isnull().sum() > 0:
        null_count = df[col].isnull().sum()
        df[col].fillna('Unknown', inplace=True)
        categorical_nulls_filled += null_count
        print(f"Filled {col} nulls with 'Unknown' ({null_count} nulls)")

Filled Initial_CAD_Event_Type nulls with 'Unknown' (1 nulls)
Filled Incident_Station_Area nulls with 'Unknown' (1 nulls)
Filled Property_Use nulls with 'Unknown' (224 nulls)
Filled Method_Of_Fire_Control nulls with 'Unknown' (10598 nulls)
Filled Area_of_Origin nulls with 'Unknown' (10594 nulls)
Filled Extent_Of_Fire nulls with 'Unknown' (18487 nulls)
Filled Ignition_Source nulls with 'Unknown' (10594 nulls)
Filled Material_First_Ignited nulls with 'Unknown' (10641 nulls)
Filled Possible_Cause nulls with 'Unknown' (10594 nulls)
Filled Smoke_Alarm_at_Fire_Origin nulls with 'Unknown' (18486 nulls)
Filled Sprinkler_System_Presence nulls with 'Unknown' (18486 nulls)


C:\Users\Ayush\AppData\Local\Temp\ipykernel_6240\3594084696.py:6: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df[col].fillna('Unknown', inplace=True)


In [30]:
# Flag columns with > 40% nulls BEFORE filling (report what was high)
print("\nChecking for columns with > 40% null values (before filling):")
high_null_columns = []
original_df_check = pd.read_csv('../data/fire_incidents_data.csv')
null_counts = pd.isnull(original_df_check).sum()
for col, count in null_counts.items():
    percentage = (count / len(original_df_check)) * 100
    if percentage > 40:
        high_null_columns.append(col)
        print(f"⚠️  WARNING: Column '{col}' has {percentage:.2f}% null values")

if not high_null_columns:
    print("✓ No columns with > 40% null values found")


Checking for columns with > 40% null values (before filling):
⚠️  WARNING: Column 'Exposures' has 69.52% null values
⚠️  WARNING: Column 'Building_Status' has 50.55% null values
⚠️  WARNING: Column 'Estimated_Number_Of_Persons_Displaced' has 50.56% null values
⚠️  WARNING: Column 'Business_Impact' has 50.56% null values
⚠️  WARNING: Column 'Level_Of_Origin' has 50.56% null values
⚠️  WARNING: Column 'Extent_Of_Fire' has 50.56% null values
⚠️  WARNING: Column 'Smoke_Spread' has 50.56% null values
⚠️  WARNING: Column 'Smoke_Alarm_at_Fire_Origin' has 50.56% null values
⚠️  WARNING: Column 'Smoke_Alarm_at_Fire_Origin_Alarm_Failure' has 50.56% null values
⚠️  WARNING: Column 'Smoke_Alarm_at_Fire_Origin_Alarm_Type' has 50.56% null values
⚠️  WARNING: Column 'Smoke_Alarm_Impact_on_Persons_Evacuating_Impact_on_Evacuation' has 50.56% null values
⚠️  WARNING: Column 'Fire_Alarm_System_Presence' has 50.56% null values
⚠️  WARNING: Column 'Fire_Alarm_System_Operation' has 50.56% null values
⚠️  W

C:\Users\Ayush\AppData\Local\Temp\ipykernel_6240\4213849840.py:4: DtypeWarning: Columns (0: Incident_Station_Area) have mixed types. Specify dtype option on import or set low_memory=False.
  original_df_check = pd.read_csv('../data/fire_incidents_data.csv')


## 5. Remove Outliers

Handle outliers in numeric columns:
- `response_time_min`: cap negative values at 0 and values above 99th percentile
- `Estimated_Dollar_Loss`: cap values above 99th percentile using IQR method

Note: Values are capped but rows are NOT removed.

In [31]:
print("\n" + "="*80)
print("5. REMOVING OUTLIERS (CAPPING EXTREME VALUES)")
print("="*80)

# Handle response_time_min outliers
if 'response_time_min' in df.columns:
    # Cap negative values at 0 (error handling)
    neg_count = (df['response_time_min'] < 0).sum()
    df.loc[df['response_time_min'] < 0, 'response_time_min'] = 0
    if neg_count > 0:
        print(f"Capped {neg_count} negative response_time_min values to 0")
    
    # Cap values above 99th percentile
    p99_response = df['response_time_min'].quantile(0.99)
    above_p99 = (df['response_time_min'] > p99_response).sum()
    df.loc[df['response_time_min'] > p99_response, 'response_time_min'] = p99_response
    if above_p99 > 0:
        print(f"Capped {above_p99} response_time_min values above 99th percentile ({p99_response:.2f} min)")


5. REMOVING OUTLIERS (CAPPING EXTREME VALUES)
Capped 2 negative response_time_min values to 0
Capped 366 response_time_min values above 99th percentile (11.79 min)


In [32]:
# Handle Estimated_Dollar_Loss outliers (IQR method)
if 'Estimated_Dollar_Loss' in df.columns:
    Q1 = df['Estimated_Dollar_Loss'].quantile(0.25)
    Q3 = df['Estimated_Dollar_Loss'].quantile(0.75)
    IQR = Q3 - Q1
    upper_whisker = Q3 + (1.5 * IQR)
    p99_loss = df['Estimated_Dollar_Loss'].quantile(0.99)
    
    # Use the 99th percentile as the cap
    above_p99_loss = (df['Estimated_Dollar_Loss'] > p99_loss).sum()
    df.loc[df['Estimated_Dollar_Loss'] > p99_loss, 'Estimated_Dollar_Loss'] = p99_loss
    if above_p99_loss > 0:
        print(f"Capped {above_p99_loss} Estimated_Dollar_Loss values above 99th percentile (${p99_loss:,.2f})")

Capped 199 Estimated_Dollar_Loss values above 99th percentile ($500,000.00)


## 6. Validate Geolocation

Filter to only valid Toronto coordinates:
- **Latitude:** 43.58 to 43.86
- **Longitude:** -79.64 to -79.12

Track how many rows are removed due to geolocation validation.

In [39]:
print("\n" + "="*80)
print("6. VALIDATING GEOLOCATION (TORONTO BOUNDS)")
print("="*80)

initial_rows = len(df)

# Define Toronto geolocation bounds
lat_min, lat_max = 43.58, 43.86
lon_min, lon_max = -79.64, -79.12

# Filter for valid coordinates
df = df[
    (df['Latitude'] >= lat_min) & (df['Latitude'] <= lat_max) &
    (df['Longitude'] >= lon_min) & (df['Longitude'] <= lon_max)
]

rows_dropped = initial_rows - len(df)
print(f"\nRows before geolocation validation: {initial_rows}")
print(f"Rows after geolocation validation: {len(df)}")
print(f"Rows dropped due to invalid coordinates: {rows_dropped}")
print(f"Latitude bounds: {lat_min} to {lat_max}")
print(f"Longitude bounds: {lon_min} to {lon_max}")


6. VALIDATING GEOLOCATION (TORONTO BOUNDS)

Rows before geolocation validation: 36559
Rows after geolocation validation: 36559
Rows dropped due to invalid coordinates: 0
Latitude bounds: 43.58 to 43.86
Longitude bounds: -79.64 to -79.12


## 7. Save Cleaned Output

Write the cleaned DataFrame to a CSV file for downstream analysis.

In [40]:
print("\n" + "="*80)
print("7. SAVING CLEANED OUTPUT")
print("="*80)

output_path = '../outputs/fire_incidents_cleaned.csv'
df.to_csv(output_path, index=False)
print(f"\n✓ Cleaned dataset saved to: {output_path}")
print(f"Final dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")


7. SAVING CLEANED OUTPUT

✓ Cleaned dataset saved to: ../outputs/fire_incidents_cleaned.csv
Final dataset shape: 36559 rows × 49 columns


## 8. Write Cleaning Log

Create a summary of the cleaning operations performed and save to a markdown log file for documentation and audit purposes.

In [41]:
print("\n" + "="*80)
print("8. WRITING CLEANING LOG")
print("="*80)

# Reload original data to calculate statistics
# We reload the original CSV to compare row/column counts before and after cleaning
original_df = pd.read_csv('../data/fire_incidents_data.csv')
original_rows = len(original_df)
final_rows = len(df)

# Calculate null metrics before and after cleaning
# Count nulls in original dataframe (all 43 columns)
original_total_nulls = original_df.isnull().sum().sum()

# Count nulls only in the original 43 columns (before new columns were added)
# This shows nulls we actually filled in existing columns
original_cols_list = original_df.columns.tolist()
nulls_filled_in_original_cols = original_df[original_cols_list].isnull().sum().sum() - df[original_cols_list].isnull().sum().sum()

# Count nulls in newly added columns
new_cols = ['response_time_min', 'control_time_min', 'year', 'month', 'hour_of_day', 'day_of_week']
nulls_in_new_cols = df[new_cols].isnull().sum().sum()

# Count total nulls in final dataset
cleaned_total_nulls = df.isnull().sum().sum()

# Prepare cleaning summary
# This log captures all transformations applied during the cleaning process
# for documentation, audit trails, and understanding data quality improvements
log_content = f"""# Data Cleaning Log
**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## Summary Statistics
- **Original Row Count:** {original_rows:,}
- **Final Row Count:** {final_rows:,}
- **Rows Removed:** {original_rows - final_rows:,} ({((original_rows - final_rows) / original_rows * 100):.2f}%)
- **Original Columns:** {len(original_df.columns)}
- **Final Columns:** {len(df.columns)}

## Columns Added
The following derived and computed columns were added during cleaning:
- `response_time_min`: Response time in minutes (TFS_Arrival_Time - TFS_Alarm_Time)
  - Calculated as: (Arrival Time - Alarm Time) / 60 seconds per minute
  - Useful for analyzing emergency response efficiency
- `control_time_min`: Fire control time in minutes (Fire_Under_Control_Time - TFS_Alarm_Time)
  - Calculated as: (Fire Control Time - Alarm Time) / 60 seconds per minute
  - Useful for analyzing fire suppression effectiveness
- `year`: Year extracted from TFS_Alarm_Time (useful for trend analysis across years)
- `month`: Month extracted from TFS_Alarm_Time (useful for seasonal analysis)
- `hour_of_day`: Hour of day extracted from TFS_Alarm_Time (useful for temporal patterns)
- `day_of_week`: Day of week extracted from TFS_Alarm_Time (Monday=0, Sunday=6) (useful for weekly patterns)

## Datetime Columns Parsed
The following columns were converted from string to datetime format:
This allows for time-based calculations and temporal analysis on fire incidents.
- TFS_Alarm_Time (when the alarm was first triggered)
- TFS_Arrival_Time (when firefighters arrived at the scene)
- Ext_agent_app_or_defer_time (when external agent was applied or deferred)
- Fire_Under_Control_Time (when the fire was brought under control)
- Last_TFS_Unit_Clear_Time (when the last TFS unit cleared the scene)

## Categorical Columns Cleaned
The following columns had numeric code prefixes removed (e.g., "01 - Fire" → "Fire"):
This standardization improves readability and makes categorical analysis cleaner.
- Final_Incident_Type
- Initial_CAD_Event_Type
- Property_Use
- Area_of_Origin
- Possible_Cause
- Method_Of_Fire_Control
- Extent_Of_Fire
- Ignition_Source
- Material_First_Ignited
- Smoke_Alarm_at_Fire_Origin
- Sprinkler_System_Presence

## Nulls Filled

### Numeric Columns (Filled with 0)
These columns represent counts and casualties, so missing values logically represent zero occurrences:
- Civilian_Casualties (no civilians injured = 0)
- TFS_Firefighter_Casualties (no firefighters injured = 0)
- Count_of_Persons_Rescued (no persons rescued = 0)
- Estimated_Number_Of_Persons_Displaced (no persons displaced = 0)

### Other Numeric Columns
- Estimated_Dollar_Loss: Filled with median value
  - Reasoning: Loss amounts that are missing are imputed with the median to preserve statistical distribution
  - Median is preferred over mean to reduce impact of extreme outliers

### Categorical Columns
All remaining null values in categorical columns were filled with "Unknown"
This preserves row integrity while flagging where data was incomplete.

**Nulls Filled in Original Columns:** {nulls_filled_in_original_cols:,}
**Nulls in Newly Added Columns:** {nulls_in_new_cols:,} (from extracted temporal features)
**Net Change in Total Nulls:** {original_total_nulls - cleaned_total_nulls:,}

## High-Null Columns (> 40%)
The following columns had more than 40% null values in the original dataset:
These are flagged for awareness as they may have limited analytical value:
"""

# Add high-null column info
null_counts_original = original_df.isnull().sum()
for col, count in null_counts_original.items():
    percentage = (count / len(original_df)) * 100
    if percentage > 40:
        log_content += f"- **{col}**: {percentage:.2f}% null ({count:,} of {len(original_df):,} rows)\n"

nulls_over_40 = [(col, (count / len(original_df)) * 100) for col, count in null_counts_original.items() if (count / len(original_df)) * 100 > 40]
if not nulls_over_40:
    log_content += "- None\n"

# Add outlier capping info
log_content += f"""
## Outliers Capped
Extreme values were capped at the 99th percentile to handle data entry errors or exceptional cases:

- **response_time_min**: Negative values capped to 0 (impossible negative response times); values above 99th percentile ({df['response_time_min'].quantile(0.99):.2f} min) capped
  - Reasoning: Response times should be non-negative. Extreme outliers may indicate data quality issues.

- **Estimated_Dollar_Loss**: Values above 99th percentile (${df['Estimated_Dollar_Loss'].quantile(0.99):,.2f}) capped using 99th percentile method
  - Reasoning: Extreme loss values can skew statistical analyses. Capping preserves the value while preventing extreme influence.

## Geolocation Validation
Only records with coordinates within the Toronto city boundaries were retained:
- **Latitude Range:** 43.58 to 43.86
- **Longitude Range:** -79.64 to -79.12
- **Rows Dropped:** {rows_dropped:,} (records with coordinates outside Toronto bounds)
- Reasoning: Removes out-of-jurisdiction incidents and potential data entry errors.

## Cleaning Completion
All data cleaning steps completed successfully!
Output saved to: `outputs/fire_incidents_cleaned.csv`
"""

# Write log to file with UTF-8 encoding
# UTF-8 encoding is specified to handle special Unicode characters (checkmarks, emojis, etc.)
# without raising UnicodeEncodeError on Windows systems that default to cp1252 encoding
log_path = '../outputs/cleaning_log.md'
with open(log_path, 'w', encoding='utf-8') as f:
    f.write(log_content)

print(f"\n[SUCCESS] Cleaning log saved to: {log_path}")


8. WRITING CLEANING LOG

[SUCCESS] Cleaning log saved to: ../outputs/cleaning_log.md


C:\Users\Ayush\AppData\Local\Temp\ipykernel_6240\1686565884.py:7: DtypeWarning: Columns (0: Incident_Station_Area) have mixed types. Specify dtype option on import or set low_memory=False.
  original_df = pd.read_csv('../data/fire_incidents_data.csv')


## Final Summary

Data cleaning is complete!

In [18]:
print("\n" + "="*80)
print("DATA CLEANING COMPLETE")
print("="*80)
print(f"\n✓ Original dataset: {original_rows:,} rows × {len(original_df.columns)} columns")
print(f"✓ Cleaned dataset:  {final_rows:,} rows × {len(df.columns)} columns")
print(f"✓ Data quality improvements applied: datetime parsing, categorical cleaning,")
print(f"  missing value handling, outlier capping, and geolocation validation")
print(f"\n✓ Output files:")
print(f"  - {output_path}")
print(f"  - {log_path}")
print("\n" + "="*80)


DATA CLEANING COMPLETE

✓ Original dataset: 36,564 rows × 43 columns
✓ Cleaned dataset:  36,559 rows × 49 columns
✓ Data quality improvements applied: datetime parsing, categorical cleaning,
  missing value handling, outlier capping, and geolocation validation

✓ Output files:
  - ../outputs/fire_incidents_cleaned.csv
  - ../outputs/cleaning_log.md

